In [23]:
!pip install yfinance plotly scipy scikit-learn --quiet

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import yfinance as yf
import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import zscore
from datetime import datetime, timedelta

In [ ]:
TICKERS = [
    "AAPL", "MSFT", "NVDA", "GOOGL", "AMZN",
    "META", "TSLA", "JPM", "V", "UNH",
    "HD", "PG", "MA", "XOM", "LLY",
    "AVGO", "COST", "PEP", "KO", "MRK"
]

START_DATE = "2020-01-01"
END_DATE = datetime.today().strftime("%Y-%m-%d")

RISK_FREE_RATE = 0.04
TOP_N = 10

In [ ]:
def safe_get(dictionary, key, default=np.nan):
    """
    Safely extract values from a dictionary.
    """
    try:
        value = dictionary.get(key, default)
        return value if value is not None else default
    except Exception:
        return default


def calculate_max_drawdown(price_series):
    """
    Calculate maximum drawdown from a price series.
    """
    cumulative_max = price_series.cummax()
    drawdown = price_series / cumulative_max - 1
    return drawdown.min()


def annualized_return(price_series):
    """
    Calculate annualized return.
    """
    total_return = price_series.iloc[-1] / price_series.iloc[0] - 1
    years = len(price_series) / 252
    return (1 + total_return) ** (1 / years) - 1


def annualized_volatility(return_series):
    """
    Calculate annualized volatility.
    """
    return return_series.std() * np.sqrt(252)


def sharpe_ratio(ann_return, ann_vol, risk_free_rate=0.04):
    """
    Calculate Sharpe ratio.
    """
    if ann_vol == 0 or np.isnan(ann_vol):
        return np.nan
    return (ann_return - risk_free_rate) / ann_vol


def winsorize_series(series, lower=0.05, upper=0.95):
    """
    Reduce outlier impact by clipping extreme values.
    """
    lower_bound = series.quantile(lower)
    upper_bound = series.quantile(upper)
    return series.clip(lower_bound, upper_bound)


def zscore_clean(series):
    """
    Calculate z-score while handling missing values.
    """
    series = series.replace([np.inf, -np.inf], np.nan)
    series = series.fillna(series.median())
    return pd.Series(zscore(series), index=series.index)

In [ ]:
def download_price_data(tickers, start_date, end_date):
    """
    Download adjusted close prices from Yahoo Finance.
    """
    try:
        data = yf.download(
            tickers,
            start=start_date,
            end=end_date,
            auto_adjust=True,
            progress=False
        )

        if "Close" in data.columns:
            prices = data["Close"]
        else:
            prices = data

        prices = prices.dropna(axis=1, how="all")
        return prices

    except Exception as error:
        raise RuntimeError(f"Price download failed: {error}")


prices = download_price_data(TICKERS, START_DATE, END_DATE)

print("Price data shape:", prices.shape)
prices.tail()

Price data shape: (1622, 20)


Ticker,AAPL,AMZN,AVGO,COST,GOOGL,HD,JPM,KO,LLY,MA,META,MRK,MSFT,NVDA,PEP,PG,TSLA,UNH,V,XOM
Date,,,,,,,,,,,,,,,,,,,,
2026-06-10,291.579987,238.000000,372.100006,983.369995,356.380005,318.920013,309.140015,83.053772,1136.369995,489.079987,570.451294,118.239708,397.359985,200.419998,144.320007,149.050003,381.589996,405.146027,322.959991,150.619995
2026-06-11,295.630005,241.509995,385.570007,975.690002,357.769989,326.010010,313.489990,82.000580,1160.949951,486.510010,567.903625,119.897789,390.339996,204.869995,143.729996,148.339996,399.149994,403.246857,319.049988,146.600006
2026-06-12,291.130005,238.550003,382.070007,982.349976,359.679993,328.390015,320.720001,82.090004,1133.000000,489.980011,566.454956,118.200005,390.739990,205.190002,144.270004,149.610001,406.429993,406.200012,322.390015,147.009995
2026-06-15,296.420013,246.020004,393.940002,979.450012,369.350006,329.820007,319.399994,80.910004,1129.349976,490.640015,593.479980,114.900002,399.760010,212.449997,146.250000,150.460007,411.149994,411.040009,323.820007,140.919998
2026-06-16,299.239990,246.000000,376.709991,986.679993,373.250000,337.089996,331.140015,80.279999,1122.500000,501.329987,600.210022,115.169998,393.829987,207.410004,146.119995,152.490005,404.660004,407.649994,333.119995,141.860001


In [ ]:
def download_fundamental_data(tickers):
    """
    Download fundamental data from Yahoo Finance.
    """
    records = []

    for ticker in tickers:
        try:
            stock = yf.Ticker(ticker)
            info = stock.info

            records.append({
                "Ticker": ticker,
                "Sector": safe_get(info, "sector", "Unknown"),
                "Market Cap": safe_get(info, "marketCap"),
                "PE Ratio": safe_get(info, "trailingPE"),
                "PB Ratio": safe_get(info, "priceToBook"),
                "ROE": safe_get(info, "returnOnEquity"),
                "Profit Margin": safe_get(info, "profitMargins"),
                "Debt to Equity": safe_get(info, "debtToEquity"),
                "Beta": safe_get(info, "beta")
            })

        except Exception as error:
            print(f"Warning: Could not download fundamentals for {ticker}: {error}")

    return pd.DataFrame(records).set_index("Ticker")


fundamentals = download_fundamental_data(TICKERS)

fundamentals

,Sector,Market Cap,PE Ratio,PB Ratio,ROE,Profit Margin,Debt to Equity,Beta
Ticker,,,,,,,,
AAPL,Technology,4346723172352,35.829300,40.764460,1.41471,0.27152,79.548,1.086
MSFT,Technology,2814708285440,22.581049,6.793302,0.34014,0.39342,30.271,1.103
NVDA,Technology,4956827418624,31.339968,25.359356,1.14288,0.62966,6.555,2.202
GOOGL,Communication Services,4439174283264,27.770230,9.206611,0.38885,0.37919,20.026,1.237
AMZN,Consumer Cyclical,2554813480960,30.566280,5.779573,0.24285,0.12224,53.300,1.444
META,Communication Services,1440758366208,20.646782,5.911491,0.32930,0.32837,35.608,1.229
TSLA,Consumer Cyclical,1488693755904,363.651370,18.102024,0.04901,0.03946,18.738,1.798
JPM,Financial Services,893509828608,15.962662,2.597465,0.16465,0.33936,NaN,1.000
V,Financial Services,628298088448,28.828970,17.722347,0.60349,0.51679,67.233,0.765


In [ ]:
def build_factor_features(prices, fundamentals):
    """
    Create momentum, risk, value, and quality features.
    """
    returns = prices.pct_change().dropna()

    feature_rows = []

    for ticker in prices.columns:
        try:
            price_series = prices[ticker].dropna()
            return_series = returns[ticker].dropna()

            if len(price_series) < 252:
                continue

            latest_price = price_series.iloc[-1]

            mom_3m = latest_price / price_series.iloc[-63] - 1
            mom_6m = latest_price / price_series.iloc[-126] - 1
            mom_12m = latest_price / price_series.iloc[-252] - 1

            ann_ret = annualized_return(price_series)
            ann_vol = annualized_volatility(return_series)
            max_dd = calculate_max_drawdown(price_series)
            sharpe = sharpe_ratio(ann_ret, ann_vol, RISK_FREE_RATE)

            row = {
                "Ticker": ticker,
                "Momentum 3M": mom_3m,
                "Momentum 6M": mom_6m,
                "Momentum 12M": mom_12m,
                "Annualized Return": ann_ret,
                "Annualized Volatility": ann_vol,
                "Max Drawdown": max_dd,
                "Sharpe Ratio": sharpe
            }

            if ticker in fundamentals.index:
                for col in fundamentals.columns:
                    row[col] = fundamentals.loc[ticker, col]

            feature_rows.append(row)

        except Exception as error:
            print(f"Warning: Feature engineering failed for {ticker}: {error}")

    features = pd.DataFrame(feature_rows).set_index("Ticker")
    return features


features = build_factor_features(prices, fundamentals)

features.head()

,Momentum 3M,Momentum 6M,Momentum 12M,Annualized Return,Annualized Volatility,Max Drawdown,Sharpe Ratio,Sector,Market Cap,PE Ratio,PB Ratio,ROE,Profit Margin,Debt to Equity,Beta
Ticker,,,,,,,,,,,,,,,
AAPL,0.198350,0.093707,0.514097,0.246836,0.313435,-0.333605,0.659902,Technology,4346723172352,35.829300,40.764460,1.41471,0.27152,79.548,1.086
AMZN,0.172154,0.105419,0.138362,0.159495,0.353587,-0.561453,0.337950,Consumer Cyclical,2554813480960,30.566280,5.779573,0.24285,0.12224,53.300,1.444
AVGO,0.194886,0.113041,0.506396,0.500251,0.441608,-0.483000,1.042216,Technology,1869253312512,65.157540,21.318502,0.37281,0.38848,74.018,1.433
COST,0.008360,0.149788,0.007999,0.226060,0.235496,-0.314024,0.790076,Consumer Defensive,428218712064,48.693394,25.902410,0.29152,0.03010,60.257,0.868
GOOGL,0.213796,0.212562,1.117524,0.303336,0.323352,-0.443201,0.814394,Communication Services,4439174283264,27.770230,9.206611,0.38885,0.37919,20.026,1.237


In [ ]:
def build_factor_scores(features):
    """
    Build normalized factor scores.
    Higher score = better.
    """
    df = features.copy()

    numeric_cols = df.select_dtypes(include=[np.number]).columns

    for col in numeric_cols:
        df[col] = winsorize_series(df[col])

    scores = pd.DataFrame(index=df.index)

    # Momentum: higher is better
    scores["Momentum Score"] = (
        zscore_clean(df["Momentum 3M"]) +
        zscore_clean(df["Momentum 6M"]) +
        zscore_clean(df["Momentum 12M"])
    ) / 3

    # Value: lower PE, PB, and size valuation are better
    scores["Value Score"] = (
        -zscore_clean(df["PE Ratio"]) +
        -zscore_clean(df["PB Ratio"])
    ) / 2

    # Quality: higher ROE and margin are better, lower debt is better
    scores["Quality Score"] = (
        zscore_clean(df["ROE"]) +
        zscore_clean(df["Profit Margin"]) +
        -zscore_clean(df["Debt to Equity"])
    ) / 3

    # Risk: lower volatility, lower beta, lower drawdown are better
    scores["Risk Score"] = (
        -zscore_clean(df["Annualized Volatility"]) +
        -zscore_clean(df["Beta"]) +
        zscore_clean(df["Max Drawdown"])
    ) / 3

    scores["Composite Score"] = (
        0.30 * scores["Momentum Score"] +
        0.25 * scores["Value Score"] +
        0.25 * scores["Quality Score"] +
        0.20 * scores["Risk Score"]
    )

    result = df.join(scores)
    result["Rank"] = result["Composite Score"].rank(ascending=False)

    return result.sort_values("Composite Score", ascending=False)


factor_results = build_factor_scores(features)

factor_results.head(10)

,Momentum 3M,Momentum 6M,Momentum 12M,Annualized Return,Annualized Volatility,Max Drawdown,Sharpe Ratio,Sector,Market Cap,PE Ratio,...,ROE,Profit Margin,Debt to Equity,Beta,Momentum Score,Value Score,Quality Score,Risk Score,Composite Score,Rank
Ticker,,,,,,,,,,,,,,,,,,,,,
GOOGL,0.213796,0.212562,0.544269,0.303336,0.323352,-0.443201,0.814394,Communication Services,4439174283264,27.770230,...,0.38885,0.379190,20.0260,1.23700,1.380437,0.442151,0.445015,-0.186820,0.598559,1.0
JPM,0.156720,0.044735,0.248632,0.173215,0.309446,-0.436265,0.430493,Financial Services,893509828608,20.412576,...,0.16465,0.339360,NaN,1.00000,0.272986,0.959545,0.006381,0.045610,0.332499,2.0
KO,0.063556,0.146314,0.170641,0.093569,0.206584,-0.369875,0.265334,Consumer Defensive,343897407488,25.135220,...,0.43372,0.278000,124.9430,0.35400,0.113835,0.487571,-0.147693,1.031065,0.325333,3.0
UNH,0.235660,0.211274,0.360488,0.071284,0.342370,-0.613909,0.091373,Healthcare,362830921728,30.062452,...,0.12176,0.029934,73.9820,0.65000,1.189063,0.593166,-0.686234,-0.263568,0.280738,4.0
MRK,0.013085,0.165543,0.470113,0.076692,0.248770,-0.434434,0.147492,Healthcare,285116563456,32.518310,...,0.18944,0.135860,106.9360,0.21800,0.420993,0.407989,-0.549699,0.814975,0.253865,5.0
PG,0.047224,0.066141,-0.024884,0.060101,0.206831,-0.299952,0.097186,Consumer Defensive,350593875968,21.979563,...,0.31112,0.191620,67.6510,0.38500,-0.454636,0.745356,-0.187946,1.179072,0.238776,6.0
AAPL,0.198350,0.093707,0.514097,0.246836,0.313435,-0.333605,0.659902,Technology,4346723172352,35.829300,...,1.41471,0.271520,79.5480,1.08600,0.945029,-1.128935,0.750497,0.221549,0.233209,7.0
LLY,0.224805,0.060365,0.399788,0.410661,0.344297,-0.344750,1.076571,Healthcare,991616434176,39.474617,...,1.07459,0.349860,139.0150,0.51700,0.779949,-0.881742,0.420999,0.478249,0.214449,8.0
XOM,-0.041083,0.212980,0.301404,0.165463,0.326409,-0.550049,0.384374,Energy,583359922176,23.693604,...,0.09873,0.077650,18.2610,0.21455,0.116972,0.851225,-0.383226,0.253257,0.202743,9.0


In [ ]:
top_stocks = factor_results.head(TOP_N)

display_columns = [
    "Sector",
    "Momentum Score",
    "Value Score",
    "Quality Score",
    "Risk Score",
    "Composite Score",
    "Annualized Return",
    "Annualized Volatility",
    "Sharpe Ratio",
    "Max Drawdown",
    "Rank"
]

top_stocks[display_columns]

,Sector,Momentum Score,Value Score,Quality Score,Risk Score,Composite Score,Annualized Return,Annualized Volatility,Sharpe Ratio,Max Drawdown,Rank
Ticker,,,,,,,,,,,
GOOGL,Communication Services,1.380437,0.442151,0.445015,-0.186820,0.598559,0.303336,0.323352,0.814394,-0.443201,1.0
JPM,Financial Services,0.272986,0.959545,0.006381,0.045610,0.332499,0.173215,0.309446,0.430493,-0.436265,2.0
KO,Consumer Defensive,0.113835,0.487571,-0.147693,1.031065,0.325333,0.093569,0.206584,0.265334,-0.369875,3.0
UNH,Healthcare,1.189063,0.593166,-0.686234,-0.263568,0.280738,0.071284,0.342370,0.091373,-0.613909,4.0
MRK,Healthcare,0.420993,0.407989,-0.549699,0.814975,0.253865,0.076692,0.248770,0.147492,-0.434434,5.0
PG,Consumer Defensive,-0.454636,0.745356,-0.187946,1.179072,0.238776,0.060101,0.206831,0.097186,-0.299952,6.0
AAPL,Technology,0.945029,-1.128935,0.750497,0.221549,0.233209,0.246836,0.313435,0.659902,-0.333605,7.0
LLY,Healthcare,0.779949,-0.881742,0.420999,0.478249,0.214449,0.410661,0.344297,1.076571,-0.344750,8.0
XOM,Energy,0.116972,0.851225,-0.383226,0.253257,0.202743,0.165463,0.326409,0.384374,-0.550049,9.0


In [ ]:
fig = px.bar(
    factor_results.head(15).reset_index(),
    x="Ticker",
    y="Composite Score",
    title="Top 15 Stocks by Composite Factor Score",
    text="Composite Score"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")
fig.update_layout(height=500)

fig.show()

In [ ]:
heatmap_data = factor_results[
    ["Momentum Score", "Value Score", "Quality Score", "Risk Score"]
].head(15)

fig = px.imshow(
    heatmap_data,
    text_auto=".2f",
    title="Factor Score Heatmap — Top 15 Stocks",
    aspect="auto"
)

fig.update_layout(height=600)

fig.show()

In [ ]:
plot_df = factor_results.reset_index().copy()

# Plotly marker size must be positive
plot_df["Bubble Size"] = plot_df["Composite Score"] - plot_df["Composite Score"].min() + 0.1

fig = px.scatter(
    plot_df,
    x="Annualized Volatility",
    y="Annualized Return",
    size="Bubble Size",
    color="Sector",
    hover_name="Ticker",
    hover_data={
        "Composite Score": ":.2f",
        "Bubble Size": False
    },
    title="Risk vs Return by Stock"
)

fig.update_layout(height=600)

fig.show()

In [ ]:
sector_counts = top_stocks["Sector"].value_counts().reset_index()
sector_counts.columns = ["Sector", "Count"]

fig = px.pie(
    sector_counts,
    names="Sector",
    values="Count",
    title=f"Sector Exposure of Top {TOP_N} Factor Stocks"
)

fig.show()

In [ ]:
def backtest_equal_weight_portfolio(prices, selected_tickers):
    """
    Simple equal-weighted portfolio backtest.
    """
    selected_prices = prices[selected_tickers].dropna()

    daily_returns = selected_prices.pct_change().dropna()
    portfolio_returns = daily_returns.mean(axis=1)

    equity_curve = (1 + portfolio_returns).cumprod()

    return portfolio_returns, equity_curve


selected_tickers = top_stocks.index.tolist()

portfolio_returns, equity_curve = backtest_equal_weight_portfolio(
    prices,
    selected_tickers
)

equity_curve.tail()

,0
Date,
2026-06-10,4.429428
2026-06-11,4.447543
2026-06-12,4.445888
2026-06-15,4.448990
2026-06-16,4.463902


In [ ]:
fig = px.line(
    equity_curve,
    title=f"Equal-Weighted Portfolio Equity Curve — Top {TOP_N} Factor Stocks"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Growth of $1"
)

fig.show()

In [ ]:
portfolio_drawdown = equity_curve / equity_curve.cummax() - 1

fig = px.area(
    portfolio_drawdown,
    title="Portfolio Drawdown"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Drawdown"
)

fig.show()

In [ ]:
def performance_summary(returns, equity_curve):
    """
    Calculate portfolio performance metrics.
    """
    total_return = equity_curve.iloc[-1] - 1
    ann_return = (1 + total_return) ** (252 / len(returns)) - 1
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = sharpe_ratio(ann_return, ann_vol, RISK_FREE_RATE)
    max_dd = (equity_curve / equity_curve.cummax() - 1).min()
    win_rate = (returns > 0).mean()

    return pd.DataFrame({
        "Metric": [
            "Total Return",
            "Annualized Return",
            "Annualized Volatility",
            "Sharpe Ratio",
            "Maximum Drawdown",
            "Win Rate"
        ],
        "Value": [
            total_return,
            ann_return,
            ann_vol,
            sharpe,
            max_dd,
            win_rate
        ]
    })


summary = performance_summary(portfolio_returns, equity_curve)

summary

,Metric,Value
0,Total Return,3.463902
1,Annualized Return,0.261840
2,Annualized Volatility,0.194478
3,Sharpe Ratio,1.140699
4,Maximum Drawdown,-0.314295
5,Win Rate,0.558297


In [ ]:
factor_results.to_csv("factor_ranking.csv")
top_stocks.to_csv("top_factor_stocks.csv")
summary.to_csv("portfolio_performance_summary.csv", index=False)

print("Files exported:")
print("- factor_ranking.csv")
print("- top_factor_stocks.csv")
print("- portfolio_performance_summary.csv")

Files exported:
- factor_ranking.csv
- top_factor_stocks.csv
- portfolio_performance_summary.csv


In [ ]:
readme = """
# Factor Investing Dashboard

## Overview
This project builds a quantitative factor investing dashboard using Python and Yahoo Finance data.

The model ranks stocks based on:
- Momentum
- Value
- Quality
- Risk

## Features
- Downloads historical market data
- Collects fundamental data
- Engineers factor features
- Calculates normalized factor scores
- Builds a composite ranking model
- Simulates an equal-weighted portfolio
- Visualizes factor exposure and performance

## Technologies
- Python
- pandas
- numpy
- yfinance
- plotly
- scipy

## Outputs
- Factor ranking table
- Top stock list
- Portfolio performance summary
- Interactive charts
- CSV exports

## Resume Description
Built a factor investing dashboard in Python using Yahoo Finance data to rank equities based on momentum, value, quality, and risk factors. Engineered financial features, normalized factor scores, constructed a composite ranking model, simulated an equal-weighted factor portfolio, and visualized performance using professional dashboards.
"""

with open("README.md", "w") as file:
    file.write(readme)

print("README.md created.")

README.md created.


In [ ]:
print("""
PROJECT SUMMARY

This project demonstrates:
- Quantitative equity research
- Factor investing
- Portfolio construction
- Risk analytics
- Financial data engineering
- Python dashboard development

Best use:
- GitHub portfolio
- Internship applications
- Quant finance interviews
- Asset management applications
""")


PROJECT SUMMARY

This project demonstrates:
- Quantitative equity research
- Factor investing
- Portfolio construction
- Risk analytics
- Financial data engineering
- Python dashboard development

Best use:
- GitHub portfolio
- Internship applications
- Quant finance interviews
- Asset management applications

